### 1- Embedding

In [2]:
import json
from sentence_transformers import SentenceTransformer

#Charger tout les chunks
chunks = []
with open("Forensics.jsonl","r",encoding="utf-8") as f:
    for line in f :
        chunks.append(json.loads(line))

print(f"Nombre de chunks chargés: {len(chunks)}")

Nombre de chunks chargés: 76


In [ ]:
# Modèle multilingue (supporte français ET anglais)
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modèle chargé !
Dimension des vecteurs : 384
Shape du vecteur test : (384,)


In [ ]:
#Générer les embeddings sur tout les chunks:
import numpy as np

contenus = [chunk["content"] for chunk in chunks]
embeddings = model.encode(
    contenus,
    batch_size=32,
    show_progress_bar= True,
    convert_to_numpy= True
)
print(f"Shape des embeddings : {embeddings.shape}")

#Sauvegarder les embeddings en fichier numpy
np.save("embeddings.npy", embeddings)
embeddings_charges = np.load("embeddings.npy")
print(f"Shape : {embeddings_charges.shape}")


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Shape des embeddings : (76, 384)
Shape : (76, 384)


### 2- Indexation avec FAISS

In [6]:
import faiss
import numpy as np

#On normalise les vecteurs
faiss.normalize_L2(embeddings_charges)
#Créer l'index
dimension = 384
index = faiss.IndexFlatIP(dimension)
#Ajout les vecteurs
index.add(embeddings_charges)

print(f"Nombre de vecteur indexe: {index.ntotal}")
print(f"Dimension: {index.d}")

#Sauvegarder le fichier d'indexation
faiss.write_index(index, "faiss_index.bin")
print("Index sauvegardé")


Nombre de vecteur indexe: 76
Dimension: 384
Index sauvegardé


In [ ]:
#Créer l'index BM25
from rank_bm25 import BM25Okapi

# contenu + keywords
corpus_tokenise = [
    chunk["content"].lower().split() + chunk["keywords"]
    for chunk in chunks
]

# Créer l'index BM25
bm25 = BM25Okapi(corpus_tokenise)

print(f"Index BM25 créé !")
print(f"Nombre de documents : {len(corpus_tokenise)}")
print(f"Vocabulaire : {len(bm25.idf)} termes uniques")

# Test sur un terme technique forensics
scores_test = bm25.get_scores("shellbags windows artefacts".split())
top_3 = sorted(enumerate(scores_test), key=lambda x: x[1], reverse=True)[:3]
print(f"\nTest 'shellbags windows artefacts' :")
for idx, score in top_3:
    print(f"  Chunk {idx} ({chunks[idx]['id']}) → score BM25 {score:.3f}")
    print(f"  {chunks[idx]['content'][:100]}...")

# Test sur un terme CVSS
scores_test2 = bm25.get_scores("attack vector network cvss".split())
top_3_2 = sorted(enumerate(scores_test2), key=lambda x: x[1], reverse=True)[:3]
print(f"\nTest 'attack vector network cvss' :")
for idx, score in top_3_2:
    print(f"  Chunk {idx} ({chunks[idx]['id']}) → score BM25 {score:.3f}")
    print(f"  {chunks[idx]['content'][:100]}...")

Index BM25 créé !
Nombre de documents : 76
Vocabulaire : 5519 termes uniques

Test 'shellbags windows artefacts' :
  Chunk 19 (FOR_MISC-CONCEPT--020) → score BM25 6.312
  pour déduire des actions ou événements. macOS : Impressions : Contrairement à Windows qui ne trace p...
  Chunk 50 (FOR_MISC-CONCEPT--051) → score BM25 2.627
  des traces d’activité si vous les réclamez. Pour les infrastructures complètes comme Ofﬁce 365 ou Go...
  Chunk 5 (FOR_MISC-CONCEPT--006) → score BM25 2.264
  Ransomware un samedi. Presque tous les serveurs Windows de son DataCenter (DC) sont touchés. Des ser...

Test 'attack vector network cvss' :
  Chunk 59 (CVE-005) → score BM25 12.493
  of the thing that is vulnerable, which we refer to formally as the vulnerable component. Therefore, ...
  Chunk 71 (CVE-017) → score BM25 9.580
  Required to None to reflect this more serious condition in their particular environment. The list of...
  Chunk 72 (CVE-018) → score BM25 9.374
  (X). Metrics with a value of Not D

In [8]:
import pickle

# Sauvegarder l'index BM25 et aussi le corpus tokenisé
with open("bm25_index.pkl", "wb") as f:
    pickle.dump(bm25, f)

# Sauvegarder aussi le corpus tokenisé
with open("corpus_tokenise.pkl", "wb") as f:
    pickle.dump(corpus_tokenise, f)

print("Index BM25 sauvegardé → bm25_index.pkl")
print("Corpus tokenisé sauvegardé → corpus_tokenise.pkl")

Index BM25 sauvegardé → bm25_index.pkl
Corpus tokenisé sauvegardé → corpus_tokenise.pkl
